# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [25]:
# Write your code below.
%load_ext dotenv
%dotenv 


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [26]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [27]:
import os
from glob import glob

# Write your code below.
# defining directory path
price_data_dir = os.getenv("PRICE_DATA")
# making a list of parquet files names
parquet_files = glob(os.path.join(price_data_dir,"**/*.parquet"), recursive=True)

In [28]:
# read the parquet files into Dask dataFrame
dd_px=dd.read_parquet(parquet_files).set_index("ticker") 

# just checking if it is empty or not!
parquet_files [:6] 

['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet']

In [29]:
print (dd_px)

Dask DataFrame Structure:
                          Date     Open     High      Low    Close Adj Close   Volume  source   Year
npartitions=60                                                                                      
ACN             datetime64[ns]  float64  float64  float64  float64   float64  float64  string  int32
ALDX                       ...      ...      ...      ...      ...       ...      ...     ...    ...
...                        ...      ...      ...      ...      ...       ...      ...     ...    ...
ZIXI                       ...      ...      ...      ...      ...       ...      ...     ...    ...
ZIXI                       ...      ...      ...      ...      ...       ...      ...     ...    ...
Dask Name: setindex, 2 expressions
Expr=SetIndex(frame=ReadParquetFSSpec(c3a87d3), _other='ticker', options={})


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [30]:
import numpy as np
# Write your code below.
dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1 = x['Close'].shift(1),
        Adj_Close_lag_1 = x['Adj Close'] . shift(1),
        returns = (x['Close']/x['Close'].shift(1))-1,
        hi_lo_range = x['High']-x['Low']
        )
)
dd_feat

C:\Users\cheki\AppData\Local\Temp\ipykernel_29200\1884672895.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_px.groupby('ticker', group_keys=False).apply(


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
npartitions=60,,,,,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64,float64,float64
ALDX,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [31]:
# Write your code below.
# upto now we had the schema and nothing was calucated
# now it is calculating it and transfer to panda dataFrame
pd_px = dd_feat.compute() 
# adding the moving average
pd_px ['moving_ave'] =  pd_px.groupby('ticker') ['returns'].rolling(10).mean() .reset_index(level=0, drop=True)


In [32]:
print (pd_px)

             Date   Open   High    Low  Close  Adj Close      Volume  \
ticker                                                                 
ACN    2001-07-19  15.10  15.29  15.00  15.17  11.404394  34994300.0   
ACN    2001-07-20  15.05  15.05  14.80  15.01  11.284108   9238500.0   
ACN    2001-07-23  15.00  15.01  14.55  15.00  11.276587   7501000.0   
ACN    2001-07-24  14.95  14.97  14.70  14.86  11.171341   3537300.0   
ACN    2001-07-25  14.70  14.95  14.65  14.95  11.238999   4208100.0   
...           ...    ...    ...    ...    ...        ...         ...   
ZIXI   2003-06-26   4.04   4.19   3.86   4.00   4.000000    515300.0   
ZIXI   2003-06-27   4.00   4.05   3.79   3.85   3.850000    162400.0   
ZIXI   2003-06-30   3.84   4.00   3.72   3.77   3.770000    119900.0   
ZIXI   2003-07-01   3.72   3.85   3.65   3.78   3.780000    202100.0   
ZIXI   2003-07-02   3.81   3.99   3.73   3.73   3.730000     81100.0   

          source  Year  Close_lag_1  Adj_Close_lag_1   returns 

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return? no, but we needed to copute the values, doing it in DASK should have been done Lazily, calculate the moving average and then compute it. alternatively we could do the calculation and transfer to pandas' at the same time, all at once as we did here.
+ Would it have been better to do it in Dask? Why? it could have been better in Dask in case the data is too big to be computed all at once, as we do it Lazily in Dask, and could be done for the chunk of data that we need, and part by part. but for small data that can be done all at once in panda dataframe.
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.